# COMP3710 Demo 1 面试练习 Notebook

这份 notebook 按你的 GitHub 实现改写，目标是：**能解释、能指出变量、能现场改参数、能快速重跑**。

建议面试前执行 `Restart Kernel and Run All`。现场优先改标有 **练习参数** 的 cell。代码默认使用较小分辨率，CPU 也能演示；最终高分辨率输出仍以仓库中的 `demo1_fractals.py` 为准。

## 0. 环境与核心概念

- NumPy 核心对象：`ndarray`，通常在 CPU。
- PyTorch 核心对象：`Tensor`，有 `shape`、`dtype`、`device`，可在 CPU 或 CUDA GPU。
- 现代 PyTorch 默认 eager execution；需要梯度时 autograd 动态记录计算图。本项目不训练模型、不需要梯度；`@torch.no_grad()` 可避免记录 autograd 图。
- Matplotlib 显示 GPU tensor 前，需要 `.cpu().numpy()`。

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device:', device)

In [ ]:
# NumPy array 与 PyTorch tensor：数据类似，但 tensor 还能选择 device。
a_np = np.array([[1, 2], [3, 4]], dtype=np.float32)
a_t = torch.tensor(a_np, device=device)
print('NumPy :', type(a_np), a_np.shape, a_np.dtype)
print('PyTorch:', type(a_t), tuple(a_t.shape), a_t.dtype, a_t.device)
print('tensor 运算结果:\n', a_t.square())

**口头回答：** Tensor 是多维数值数组。本项目用 `H×W` tensor 同时保存所有像素的坐标、复数状态、布尔 mask 和逃逸次数，避免 Python 逐像素双重循环。

In [ ]:
def coordinate_grid(x_limits, y_limits, width, height, device):
    # xs: (width,), ys: (height,)
    xs = torch.linspace(x_limits[0], x_limits[1], width, device=device)
    ys = torch.linspace(y_limits[0], y_limits[1], height, device=device)
    # x, y 都变成 (height, width)，每个像素对应一对坐标。
    y, x = torch.meshgrid(ys, xs, indexing='ij')
    return x, y

x_demo, y_demo = coordinate_grid((-1, 1), (-1, 1), 5, 3, device)
print('x shape:', tuple(x_demo.shape), 'y shape:', tuple(y_demo.shape))
print('x grid:\n', x_demo)

## 1. Gaussian、二维正弦波与 Gabor filter

二维余弦有两种等价参数化：

$$\cos(2\pi(f_xx+f_yy))$$

以及你的仓库写法：

$$\cos(2\pi f(x\cos\theta+y\sin\theta))$$

关系是 $f_x=f\cos\theta$、$f_y=f\sin\theta$。`sqrt(f_x²+f_y²)` 控制条纹密度；二者的比例控制方向。严格地说，`(f_x,f_y)` 是条纹法向，条纹线与它垂直。

In [ ]:
# ===== 练习参数：面试时可现场修改 =====
resolution = 400
sigma = 1.25
frequency = 0.75       # cycles per coordinate unit
angle_deg = 30.0
phase = 0.0

x, y = coordinate_grid((-4, 4), (-4, 4), resolution, resolution, device)
gaussian = torch.exp(-(x.square() + y.square()) / (2 * sigma**2))

angle = math.radians(angle_deg)
f_x = frequency * math.cos(angle)
f_y = frequency * math.sin(angle)
sinusoid = torch.cos(2 * math.pi * (f_x * x + f_y * y) + phase)
gabor = gaussian * sinusoid

print(f'f_x={f_x:.4f}, f_y={f_y:.4f}, |f|={math.hypot(f_x, f_y):.4f}')
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for ax, value, title, cmap in zip(
    axes,
    [gaussian, sinusoid, gabor],
    ['2-D Gaussian', 'Oriented cosine', 'Gabor = Gaussian × cosine'],
    ['viridis', 'coolwarm', 'coolwarm']
):
    im = ax.imshow(value.detach().cpu().numpy(), origin='lower', cmap=cmap)
    ax.set_title(title)
    ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()

### Gabor 必答

- 原理：Gaussian envelope × sinusoidal carrier。
- `sigma`：控制局部作用范围；越大覆盖越宽。
- `frequency` 或 `sqrt(f_x²+f_y²)`：控制条纹密度。
- `angle` 或 `f_x:f_y`：控制方向。
- 作用：选择性响应局部的方向与空间频率，用于边缘、纹理、脊线等特征提取。

**现场练习：** 依次试 `angle_deg=60`、`frequency=1.5`。或者跳过 `frequency/angle`，直接指定 `(f_x,f_y)=(0.8,0)`、`(0,0.8)`、`(0.6,0.6)`。运行前先口头预测条纹变化。

## 2. Mandelbrot 与 Julia：`ns` / `counts`、局部放大

共同迭代公式：$z_{n+1}=z_n^2+c$。

已经按 Lab Sheet v2.31 核对：官方代码先令 `z=torch.complex(x,y)`、`zs=z.clone()`、`ns=torch.zeros_like(z)`，每轮计算 `zs_=zs*zs+z`，然后执行 `ns += (abs(zs_) < 4)`。因此 `ns` 是每像素“仍未逃逸”的累计轮数，用于颜色映射，不参与轨道状态更新。你的代码把它改名为更清楚的整数 **`counts`**，并增加 `active` mask；两者作用等价。因为官方 `zeros_like(z)` 继承复数 dtype，官方 `ns` 虽是复数 tensor，但虚部始终为 0；实际只需要整数计数。

In [ ]:
# Lab Sheet 原始变量的缩小版演示：重点看变量含义，不用于最终高分辨率输出。
Y_sheet, X_sheet = np.mgrid[-1.3:1.3:0.02, -2:1:0.02]
x_sheet = torch.tensor(X_sheet, dtype=torch.float32, device=device)
y_sheet = torch.tensor(Y_sheet, dtype=torch.float32, device=device)
z_sheet = torch.complex(x_sheet, y_sheet)  # Mandelbrot 中每像素的 c
zs_sheet = z_sheet.clone()                 # 当前轨道值
ns_sheet = torch.zeros_like(z_sheet)       # 每像素累计计数器（继承 complex dtype）
for _ in range(20):
    zs_next_sheet = zs_sheet * zs_sheet + z_sheet
    not_diverged_sheet = torch.abs(zs_next_sheet) < 4.0
    ns_sheet += not_diverged_sheet
    zs_sheet = zs_next_sheet
print('Lab Sheet ns:', tuple(ns_sheet.shape), ns_sheet.dtype,
      'imaginary max =', ns_sheet.imag.abs().max().item())
print('Mapping: sheet z→constant_c, zs→current z, zs_→candidate, ns→counts')

In [ ]:
@torch.no_grad()
def escape_counts(initial_z, constant_c, max_iterations):
    z = initial_z.clone()
    # counts 就是很多示例中命名为 ns 的变量：每像素一个整数。
    counts = torch.zeros(z.shape, dtype=torch.int32, device=z.device)
    active = torch.ones(z.shape, dtype=torch.bool, device=z.device)

    for _ in range(max_iterations):
        candidate = z.square() + constant_c
        z = torch.where(active, candidate, z)
        active = active & (z.abs() <= 2.0)
        counts += active
        if not bool(active.any()):
            break
    return counts

def plot_escape(ax, counts, max_iterations, extent, title):
    values = counts.cpu().numpy().astype(np.float32)
    values[values >= max_iterations] = np.nan
    cmap = plt.colormaps['turbo'].copy()
    cmap.set_bad('black')
    image = ax.imshow(
        values, origin='lower', extent=extent, cmap=cmap, interpolation='bilinear'
    )
    ax.set_title(title)
    ax.set_xlabel('Real axis')
    ax.set_ylabel('Imaginary axis')
    return image

### Mandelbrot 和 Julia 的代码级区别

| | 每个像素变化的量 | 固定量 | 调用方式 |
|---|---|---|---|
| Mandelbrot | `c` | `z₀=0` | `escape_counts(zeros_like(c), c, ...)` |
| Julia | `z₀` | 一个常量 `c` | `escape_counts(initial_z, julia_constant, ...)` |

In [ ]:
# ===== 练习参数：小分辨率适合现场快速重跑 =====
width, height = 420, 320
max_iterations = 180

# Mandelbrot：缩窄坐标范围就是重新计算局部 zoom。
mandelbrot_x = (-0.82, -0.72)
mandelbrot_y = (0.02, 0.12)
mx, my = coordinate_grid(mandelbrot_x, mandelbrot_y, width, height, device)
c = torch.complex(mx, my)
mandelbrot_counts = escape_counts(torch.zeros_like(c), c, max_iterations)

# Julia：每个像素是 z0，整张图固定同一个 c。
julia_x = (-1.7, 1.7)
julia_y = (-1.25, 1.25)
jx, jy = coordinate_grid(julia_x, julia_y, width, height, device)
initial_z = torch.complex(jx, jy)
julia_constant = torch.tensor(-0.4 + 0.6j, dtype=torch.complex64, device=device)
julia_counts = escape_counts(initial_z, julia_constant, max_iterations)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
im0 = plot_escape(
    axes[0], mandelbrot_counts, max_iterations,
    (*mandelbrot_x, *mandelbrot_y), 'Mandelbrot — Seahorse Valley zoom'
)
im1 = plot_escape(
    axes[1], julia_counts, max_iterations,
    (*julia_x, *julia_y), f'Julia — c = {julia_constant.item()}'
)
fig.colorbar(im0, ax=axes[0], label='Iterations before escape')
fig.colorbar(im1, ax=axes[1], label='Iterations before escape')
plt.show()

### 如何放大局部

原题要求 decreasing the `mgrid` spacing 并 zoom。放大不是 resize 旧图片，而是缩小复平面的坐标 span 后重新采样和迭代。你的代码用 `torch.linspace`，其等效 spacing 为 `(max-min)/(samples-1)`；增大 `width/height` 就等价于减小 `mgrid` step。只提高分辨率不改变视野，只有缩小坐标范围才是 zoom。深度 zoom 常需要提高 `max_iterations`，否则慢逃逸的边界点可能被误认为有界。

In [ ]:
# 把原题 mgrid step 与 GitHub full 模式的 linspace spacing 做定量对照。
sheet_dx = 0.005
repo_full_width, repo_full_height = 1400, 1000
repo_dx = (mandelbrot_x[1] - mandelbrot_x[0]) / (repo_full_width - 1)
repo_dy = (mandelbrot_y[1] - mandelbrot_y[0]) / (repo_full_height - 1)
horizontal_zoom = 3.0 / (mandelbrot_x[1] - mandelbrot_x[0])  # full x range -2..1 has width 3
print(f'Lab Sheet mgrid step: {sheet_dx:g}')
print(f'Repository full dx: {repo_dx:.3e}, dy: {repo_dy:.3e}')
print(f'Horizontal field-of-view zoom: about {horizontal_zoom:.1f}×')

In [ ]:
def zoom_bounds(center_x, center_y, span_x, width, height):
    # span_y 按像素长宽比计算，避免图形被拉伸。
    span_y = span_x * height / width
    x_limits = (center_x - span_x/2, center_x + span_x/2)
    y_limits = (center_y - span_y/2, center_y + span_y/2)
    return x_limits, y_limits

# ===== 练习：span_x 减半表示横向视野再放大 2 倍 =====
practice_x, practice_y = zoom_bounds(-0.77, 0.07, span_x=0.05, width=width, height=height)
print('new x limits:', practice_x)
print('new y limits:', practice_y)

**面试回答：** Mandelbrot 固定 `z₀=0`、每像素改变 `c`；Julia 固定 `c`、每像素改变 `z₀`。阈值 2 是二次迭代的标准逃逸半径。仍然有一个 Python 迭代循环，是因为第 n+1 步依赖第 n 步；但每一步内部同时更新整张图的所有像素。严格地说应说轨道 bounded（有界）或 escapes，不要把所有黑色点都说成 converge。Lab Sheet 中 `c=0.5+0.5i` tends to zero 的例子不成立：canonical 轨道在第 5 步的模约为 3.55，已经逃逸。

## 3. Sierpinski carpet：你的 Part 3

每层把正方形分成 `3×3` 并删除中心。坐标的三进制数字表示某层落在左/中/右；若 x 和 y 在同一层的数字都为 1，就位于中心格。代码仅对层数循环，每层用 tensor mask 同时处理全部像素。

In [ ]:
@torch.no_grad()
def sierpinski_carpet(level, device):
    size = 3**level
    coordinates = torch.arange(size, dtype=torch.int64, device=device)
    y, x = torch.meshgrid(coordinates, coordinates, indexing='ij')
    active = torch.ones((size, size), dtype=torch.bool, device=device)
    removal_depth = torch.zeros((size, size), dtype=torch.int16, device=device)
    working_x, working_y = x.clone(), y.clone()

    for step in range(1, level + 1):
        removed_now = (
            active
            & (working_x.remainder(3) == 1)
            & (working_y.remainder(3) == 1)
        )
        removal_depth[removed_now] = step
        active &= ~removed_now
        working_x.div_(3, rounding_mode='floor')
        working_y.div_(3, rounding_mode='floor')

    removal_depth[active] = level + 1
    return active, removal_depth

level = 5  # 练习参数：5 -> 6 时，边长 243 -> 729，像素数增加 9 倍
carpet, removal_depth = sierpinski_carpet(level, device)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
axes[0].imshow(carpet.cpu().numpy(), cmap='gray', origin='lower', interpolation='nearest')
axes[0].set_title(f'Sierpinski carpet, level {level}')
axes[0].axis('off')
depth_image = axes[1].imshow(
    removal_depth.cpu().numpy(), cmap='magma', origin='lower', interpolation='nearest'
)
axes[1].set_title('Removal depth')
axes[1].axis('off')
fig.colorbar(depth_image, ax=axes[1], label='Iteration removed')
plt.show()

In [ ]:
@torch.no_grad()
def box_counting_dimension(mask, level):
    size = mask.shape[0]
    box_sizes = np.array([3**power for power in range(level)], dtype=np.int64)
    counts = []
    for box_size in box_sizes:
        boxes_per_side = size // int(box_size)
        blocks = mask.reshape(
            boxes_per_side, int(box_size), boxes_per_side, int(box_size)
        )
        occupied = blocks.any(dim=3).any(dim=1)
        counts.append(int(occupied.sum().item()))
    inverse_scales = size / box_sizes.astype(np.float64)
    counts = np.array(counts, dtype=np.float64)
    dimension = float(np.polyfit(np.log(inverse_scales), np.log(counts), 1)[0])
    return inverse_scales, counts, dimension

inverse_scales, occupied_counts, estimated_d = box_counting_dimension(carpet, level)
theoretical_d = math.log(8) / math.log(3)
print(f'Estimated dimension  : {estimated_d:.6f}')
print(f'Theoretical log(8)/log(3): {theoretical_d:.6f}')

log_scale = np.log(inverse_scales)
log_count = np.log(occupied_counts)
fit = np.polyval(np.polyfit(log_scale, log_count, 1), log_scale)
plt.figure(figsize=(6, 4))
plt.scatter(log_scale, log_count, label='Measured')
plt.plot(log_scale, fit, label=f'Slope = {estimated_d:.4f}')
plt.xlabel('log(1 / box scale)')
plt.ylabel('log(occupied boxes)')
plt.title('Box-counting dimension')
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## 4. 模拟面试：看到问题后先口答，再展开答案

1. **PyTorch 和 NumPy 有什么区别？** 说 CPU/GPU、autograd、Tensor/ndarray，再指出本项目不需要梯度。
2. **Tensor 有什么作用？** 说 shape/dtype/device，以及整张图并行表示。
3. **为什么 `.cpu().numpy()`？** GPU tensor 先回 CPU，再给 NumPy/Matplotlib。
4. **`f_x`、`f_y` 怎么改变图？** 模长管密度，比例管方向；它们构成频率向量。
5. **Gabor filter 是什么？** Gaussian envelope × sinusoid；局部频率/方向选择。
6. **官方 `ns` 有什么用？** 与你的 `counts` 等价；逐轮累计 `not_diverged`，记录每像素逃逸时间，用于着色。
7. **为什么官方 `ns` 是 complex tensor？** 因为 `zeros_like(z)` 继承复数 dtype；实际计数虚部为 0，你用 int32 更合适。
8. **官方 `z`、`zs`、`zs_` 分别是什么？** `z` 是每像素 c，`zs` 是当前轨道，`zs_` 是下一步；你的命名拆成 `constant_c/current z/candidate`。
9. **`active` 有什么用？** 阻止已逃逸状态继续写回并允许全部逃逸时提前结束，避免状态溢出；但 `candidate` 仍对全图计算，不能说完全跳过 inactive 像素。
10. **如何放大局部？** 缩小坐标范围后重新采样；不是 resize。
11. **`linspace` 怎么对应原题的 mgrid spacing？** `dx=(xmax-xmin)/(width-1)`；增大样本数就是减小 spacing。
12. **提高分辨率等于放大吗？** 不等于；视野不变，只是像素更密。
13. **Mandelbrot 和 Julia 的区别？** Mandelbrot 变 c 固定 z₀；Julia 变 z₀ 固定 c。
14. **为什么官方阈值 4、你用 2？** 二次 Mandelbrot 的标准逃逸半径是 2；4 也会识别最终发散点，但逃逸颜色更晚。
15. **为什么还有 for 循环？** 时间迭代有依赖；每轮内部空间像素仍并行。
16. **PyTorch computational graph 怎么解释？** 默认 eager 执行；需要梯度时 autograd 动态记录图。本项目无训练，用 no_grad。
17. **为什么 Sierpinski substantially different？** 三进制几何删除 vs 复数逃逸动力系统。
18. **box-counting 在算什么？** 多尺度非空 box 数的 log-log 斜率；结果约 1.8928。
19. **AI 做了什么、你做了什么？** 展示 prompt/输出/修改记录，并具体说你亲自运行、改 Julia 参数、验证结果和理解代码。

### Rubric 对齐的 3 分钟陈述

- **0:00-0:25**：说明完成三个 Part、公开 GitHub、CPU/CUDA 与 quick/full workflow。
- **0:25-0:55**：展示 Gaussian、cosine、Gabor；说 envelope、carrier、frequency、orientation。
- **0:55-1:40**：展示 Mandelbrot zoom 和 Julia；量化 spacing；指出 `counts≈ns` 和两次 `escape_counts` 调用。
- **1:40-2:35**：展示 Sierpinski 的三进制 mask、PyTorch 并行、removal-depth 和 box-counting 1.892789；同时出示 Git short course 与 repository。
- **2:35-3:00**：如实说明 AI prompts、你做的修改/实验，并主动表示可以现场改参数。

Rubric 权重：functional code 20%；问答、理解与 ownership 40%；优秀理解/编程/文档/fair AI 20%；总结与 ownership 20%。所以不要把时间都花在等代码运行，必须边指代码边解释设计。

## 5. 现场修改四连练

按顺序完成，每次先预测再运行：

1. `angle_deg: 30 → 60`，再把 `frequency: 0.75 → 1.5`。
2. `span_x: 0.05 → 0.025`，说明这是再放大 2 倍。
3. `julia_constant: -0.4+0.6j → -0.8+0.156j`，比较连通/卷曲结构。
4. `level: 5 → 6`，解释边长扩大 3 倍、像素数扩大 9 倍。

最后闭眼说出：`ns ≈ counts`；Mandelbrot 的调用；Julia 的调用；Gabor 的乘法关系。